# GST Invoice Extractor on Kaggle — v2 (background jobs + live progress)

Fixes the blank screen + Cloudflare timeout: the batch is processed in the background,
the page polls for progress and then shows a highlighted table + Download Excel.

**Kaggle:** GPU T4 x2, Internet ON. Run cells 1->4. Cell 4 prints a public link.

In [1]:
# Cell 1 — install
!pip install -q -U git+https://github.com/huggingface/transformers.git accelerate pymupdf openpyxl "pillow<12" \
    fastapi "uvicorn[standard]" python-multipart nest_asyncio

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 9.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 74.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.9/121.9 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 523.9/523.9 kB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 107.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.4/71.4 kB 6.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
d

In [2]:
# Cell 2 — load GLM-OCR once
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
MODEL_PATH = "zai-org/GLM-OCR"
processor = AutoProcessor.from_pretrained(MODEL_PATH)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_PATH, torch_dtype=torch.float16, device_map="auto")
print("model ready")

preprocessor_config.json:   0%|          | 0.00/367 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.65G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/510 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/165 [00:00<?, ?B/s]

model ready


In [12]:
# Cell 3 — FastAPI app: background jobs + progress + highlighted table
import io, re, json, tempfile, shutil, uuid, threading
from pathlib import Path
from datetime import datetime
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import StreamingResponse, HTMLResponse, JSONResponse
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font
import fitz, torch

KEYS = ["GSTIN NO","Invoice no.","Invoice date","Party name",
        "Taxable value","CGST","SGST","IGST","Total"]

SCHEMA_PROMPT = (
    "Read this invoice and return ONLY one JSON object, no markdown, with exactly these keys:\n"
    '{"GSTIN NO": "<seller 15-char GSTIN>", "Invoice no.": "<string>", '
    '"Invoice date": "<DD-MM-YYYY exactly as printed>", '
    '"Party name": "<business that issued the invoice>", '
    '"Taxable value": <pre-tax amount, or null if blank>, '
    '"CGST": <number>, "SGST": <number>, '
    '"Total": <the GRAND TOTAL, never blank if a total is printed>}\n'
    "Numbers plain digits only. If a value is not printed on the invoice, return 0 — do not guess or fill it in."
)

def parse_json(t):
    t = re.sub(r"^```(?:json)?|```$", "", (t or "").strip(), flags=re.MULTILINE).strip()
    m = re.search(r"\{.*\}", t, re.DOTALL)
    return json.loads(m.group(0)) if m else None

def gstin_valid(g):
    if not g or len(g)!=15: return False
    g=g.upper(); code="0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"; f=2; s=0
    for ch in reversed(g[:14]):
        if ch not in code: return False
        cp=code.index(ch)*f; s+=cp//36+cp%36; f=1 if f==2 else 2
    return code[(36-(s%36))%36]==g[14]

def num(v):
    if v in (None,"","-"): return None
    if isinstance(v,(int,float)): return float(v)
    s=re.sub(r"[^\d.]","",str(v)); return float(s) if s else None

def norm_date(s):
    if not s: return s
    s=str(s).strip()
    for fmt in ("%d-%m-%Y","%Y-%m-%d","%d/%m/%Y","%d.%m.%Y","%d-%m-%y","%d/%m/%y","%Y/%m/%d"):
        try: return datetime.strptime(s,fmt).strftime("%d-%m-%Y")
        except ValueError: pass
    return s

def ocr_one(img_path):
    msgs=[{"role":"user","content":[{"type":"image","url":img_path},{"type":"text","text":SCHEMA_PROMPT}]}]
    inputs=processor.apply_chat_template(msgs,tokenize=True,add_generation_prompt=True,
                                         return_dict=True,return_tensors="pt").to(model.device)
    inputs.pop("token_type_ids",None)
    for k,v in inputs.items():
        if torch.is_floating_point(v): inputs[k]=v.to(model.dtype)
    with torch.inference_mode():
        gen=model.generate(**inputs,max_new_tokens=256,do_sample=False)
    out=processor.decode(gen[0][inputs["input_ids"].shape[1]:],skip_special_tokens=True)
    return parse_json(out)

def to_pages(fp, tmp):
    if fp.suffix.lower()==".pdf":
        out=[]
        for i,pg in enumerate(fitz.open(fp.as_posix())):
            p=tmp/f"{fp.stem}_{i+1:02d}.png"; pg.get_pixmap(dpi=200).save(p.as_posix()); out.append(p.as_posix())
        return out
    return [fp.as_posix()]

def validate(raw):
    rows=[]
    for i,r in enumerate(raw,1):
        status,notes="ok",[]
        if r.get("_fail"):
            rows.append({"S NO":i,**{k:None for k in KEYS},"_status":"error","_notes":"could not read"}); continue
        g=(r.get("GSTIN NO") or "").upper().replace(" ","")
        if not gstin_valid(g): status="error"; notes.append(f"GSTIN invalid: {g}")
        tx,cg,sg,ig,tot=(num(r.get(k)) for k in ["Taxable value","CGST","SGST","IGST","Total"])
        # No filling/guessing — pass through exactly what was read.
        # Flag (don't fix) anything that's missing or doesn't add up.
        if tx is None: notes.append("Taxable value blank on invoice")
        if cg is None and sg is None and ig is None:
            notes.append("no tax amounts read")
        if tot is None:
            status="error"; notes.append("Total missing")
        elif tx is not None and (cg is not None or sg is not None or ig is not None):
            check_sum = tx + (cg or 0) + (sg or 0) + (ig or 0)
            if abs(round(check_sum,2) - tot) > 1.0:
                status="error"; notes.append(f"sum {round(check_sum,2)} != total {tot}")
        if notes and status=="ok":
            status="review"
        rows.append({"S NO":i,"GSTIN NO":g,"Invoice no.":r.get("Invoice no."),
                     "Invoice date":norm_date(r.get("Invoice date")),"Party name":r.get("Party name"),
                     "Taxable value":tx,"CGST":cg,"SGST":sg,"IGST":ig,"Total":tot,
                     "_status":status,"_notes":"; ".join(notes)})
    return rows
    
def build_excel(rows):
    RED=PatternFill("solid",fgColor="F4C0C0"); AMBER=PatternFill("solid",fgColor="FAE3B0")
    wb=Workbook(); ws=wb.active; ws.title="GSTR"; ws.append(["S NO"]+KEYS+["Notes"])
    for c in ws[1]: c.font=Font(bold=True)
    for r in rows:
        ws.append([r["S NO"]]+[r[k] for k in KEYS]+[r["_notes"]])
        if r["_status"] in ("error","review"):
            fill=RED if r["_status"]=="error" else AMBER
            for c in ws[ws.max_row]: c.fill=fill
    buf=io.BytesIO(); wb.save(buf); buf.seek(0); return buf

JOBS={}

def run_job(job_id, blobs):
    tmp=Path(tempfile.mkdtemp())
    try:
        pages=[]
        for name,data in blobs:
            dest=tmp/Path(name).name; dest.write_bytes(data); pages+=to_pages(dest,tmp)
        JOBS[job_id]["total"]=len(pages)
        raw=[]
        for p in pages:
            try: rec=ocr_one(p) or {}; rec["_src"]=Path(p).name
            except Exception as e: rec={"_src":Path(p).name,"_fail":str(e)}
            raw.append(rec); JOBS[job_id]["done"]+=1
            print(f"[{job_id[:6]}] {JOBS[job_id]['done']}/{JOBS[job_id]['total']}  {Path(p).name}")
        rows=validate(raw)
        JOBS[job_id]["rows"]=rows; JOBS[job_id]["buf"]=build_excel(rows); JOBS[job_id]["status"]="done"
    except Exception as e:
        JOBS[job_id]["status"]="error"; JOBS[job_id]["error"]=str(e); print("JOB ERROR:",e)
    finally:
        shutil.rmtree(tmp,ignore_errors=True)

app=FastAPI(title="GST Invoice Extractor")

@app.post("/extract")
async def extract(files: list[UploadFile]=File(...)):
    blobs=[(f.filename, await f.read()) for f in files]
    jid=uuid.uuid4().hex
    JOBS[jid]={"total":0,"done":0,"status":"running","buf":None,"rows":None,"error":None}
    threading.Thread(target=run_job,args=(jid,blobs),daemon=True).start()
    return {"job_id":jid}

@app.get("/status/{jid}")
def status(jid: str):
    j=JOBS.get(jid)
    if not j: return {"status":"unknown"}
    return {"status":j["status"],"done":j["done"],"total":j["total"],
            "rows":j["rows"] if j["status"]=="done" else None,"error":j["error"]}

@app.get("/result/{jid}")
def result(jid: str):
    j=JOBS.get(jid)
    if not j or j["status"]!="done": return JSONResponse({"error":"not ready"},status_code=404)
    return StreamingResponse(io.BytesIO(j["buf"].getvalue()),
        media_type="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
        headers={"Content-Disposition":"attachment; filename=invoices.xlsx"})

@app.get("/",response_class=HTMLResponse)
def home():
    return """<!doctype html><html><head><meta charset=utf-8><title>GST Invoice Extractor</title>
<style>
body{font-family:system-ui,sans-serif;max-width:960px;margin:36px auto;padding:0 20px;color:#222}
h1{font-weight:600}.box{border:2px dashed #bbb;border-radius:12px;padding:22px;text-align:center}
button{margin-top:12px;padding:10px 22px;font-size:15px;border:0;border-radius:8px;background:#2563eb;color:#fff;cursor:pointer}
#prog{margin:16px 0;font-size:15px;color:#444}
table{border-collapse:collapse;width:100%;margin-top:14px;font-size:13px}
th,td{border:1px solid #e2e2e2;padding:6px 8px;text-align:left}th{background:#f3f4f6}
tr.error{background:#f8d0d0}tr.review{background:#fbe7bd}
a.dl{display:inline-block;margin:12px 0;padding:9px 18px;background:#16a34a;color:#fff;border-radius:8px;text-decoration:none}
</style></head><body>
<h1>GST Invoice -> Excel</h1>
<div class=box>
  <input type=file id=files multiple accept=.pdf,.png,.jpg,.jpeg>
  <br><button onclick=go()>Extract</button>
</div>
<div id=prog></div><div id=out></div>
<script>
const KEYS=["S NO","GSTIN NO","Invoice no.","Invoice date","Party name","Taxable value","CGST","SGST","IGST","Total","Notes"];
async function go(){
  const f=document.getElementById('files').files;
  if(!f.length){alert('Choose files first');return;}
  const fd=new FormData(); for(const x of f) fd.append('files',x);
  document.getElementById('out').innerHTML='';
  document.getElementById('prog').textContent='Uploading...';
  try{
    const r=await fetch('/extract',{method:'POST',body:fd});
    const j=await r.json(); poll(j.job_id);
  }catch(e){document.getElementById('prog').textContent='Upload failed: '+e;}
}
async function poll(id){
  try{
    const s=await (await fetch('/status/'+id)).json();
    if(s.status==='running'){
      document.getElementById('prog').textContent='Reading invoices...  '+s.done+' / '+(s.total||'?');
      setTimeout(()=>poll(id),1500); return;
    }
    if(s.status==='error'){document.getElementById('prog').textContent='Error: '+s.error;return;}
    document.getElementById('prog').textContent='Done — '+s.rows.length+' invoices.';
    render(s.rows,id);
  }catch(e){document.getElementById('prog').textContent='Lost connection: '+e;}
}
function render(rows,id){
  let h='<a class=dl href="/result/'+id+'">Download Excel</a><table><tr>';
  for(const k of KEYS) h+='<th>'+k+'</th>'; h+='</tr>';
  for(const r of rows){
    h+='<tr class="'+(r._status==='ok'?'':r._status)+'"><td>'+r["S NO"]+'</td>';
    for(let i=1;i<10;i++){const v=r[KEYS[i]];h+='<td>'+(v==null?'':v)+'</td>';}
    h+='<td>'+(r._notes||'')+'</td></tr>';
  }
  document.getElementById('out').innerHTML=h+'</table>';
}
</script></body></html>"""

print("app defined")

app defined


In [13]:
# Cell 4 — start server + public URL
import threading, subprocess, re, time, uvicorn, nest_asyncio
nest_asyncio.apply()
threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning"),
                 daemon=True).start()
time.sleep(4)
subprocess.run("wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/"
               "cloudflared-linux-amd64 -O /tmp/cf && chmod +x /tmp/cf", shell=True)
proc=subprocess.Popen(["/tmp/cf","tunnel","--url","http://localhost:8000"],
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
url=None
for line in proc.stdout:
    m=re.search(r"https://[-\w]+\.trycloudflare\.com", line)
    if m: url=m.group(0); break
threading.Thread(target=lambda:[None for _ in proc.stdout], daemon=True).start()
print("\n\n=========================================")
print(" OPEN YOUR APP:", url)
print("=========================================")

ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use




 OPEN YOUR APP: https://lending-method-vote-warranties.trycloudflare.com
[70b9b0] 1/10  10-page_01.png
[70b9b0] 2/10  10-page_02.png
[70b9b0] 3/10  10-page_03.png
[70b9b0] 4/10  10-page_04.png
[70b9b0] 5/10  10-page_05.png
[70b9b0] 6/10  10-page_06.png
[70b9b0] 7/10  10-page_07.png
[70b9b0] 8/10  10-page_08.png
[70b9b0] 9/10  10-page_09.png
[70b9b0] 10/10  10-page_10.png
[e2efef] 1/10  10-page_01.png
[e2efef] 2/10  10-page_02.png
[e2efef] 3/10  10-page_03.png
[e2efef] 4/10  10-page_04.png
[e2efef] 5/10  10-page_05.png
[e2efef] 6/10  10-page_06.png
[e2efef] 7/10  10-page_07.png
[e2efef] 8/10  10-page_08.png
[e2efef] 9/10  10-page_09.png
[e2efef] 10/10  10-page_10.png
